# In-domain final figure reproduction

This notebook retains the analysis needed for the in-domain figures that appear in the paper/supplement:

- **Figure 2B** — in-domain linear-probe validation loss and Macro-F1 curves (20 probe epochs)
- **Figure 3** — MDS of the 17-class confusion profiles
- **Supplementary Figure 1** — class-wise ΔF1 with 95% parametric bootstrap CIs
- **Supplementary Figure 3** — row-normalized confusion matrices

The downstream paper results use the **linear-probe stage only**. `history.json` files may contain later exploratory fine-tuning entries; this notebook explicitly filters to `stage == "probe"` (plus historical probe-name aliases). No fine-tuning results were evaluated or reported in the published paper; strictly linear probe only!

## Expected input layout

Set `IN_DOMAIN_RESULTS_ROOT` to a folder with this structure, or edit `RESULTS_ROOT` in the next cell:

```text
results/in_domain/
├── baseline/
│   ├── history.json
│   └── probe_confusion_matrix.csv
├── fovea-gaze/
│   ├── history.json
│   └── probe_confusion_matrix.csv
├── periph/
│   ├── history.json
│   └── probe_confusion_matrix.csv
└── periph-nf/
    ├── history.json
    └── probe_confusion_matrix.csv
```

**Important:** `probe_confusion_matrix.csv` should be the confusion matrix from the paper's **linear-probe** result. Do not substitute a later `best_confusion_matrix.csv` if that file was overwritten by exploratory fine-tuning.

Outputs default to `figure_outputs/in_domain/` and can be redirected with `IN_DOMAIN_FIGURE_OUTPUT_DIR`.


In [ ]:
from pathlib import Path
import os
import glob
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image

# -----------------------------------------------------------------------------
# Paths — edit only this section if your local layout differs.
# -----------------------------------------------------------------------------
RESULTS_ROOT = Path(
    os.environ.get("IN_DOMAIN_RESULTS_ROOT", "./results/in_domain")
).expanduser().resolve()

OUTPUT_DIR = Path(
    os.environ.get("IN_DOMAIN_FIGURE_OUTPUT_DIR", "./figure_outputs/in_domain")
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    "baseline":   RESULTS_ROOT / "baseline",
    "fovea_gaze": RESULTS_ROOT / "fovea-gaze",
    "periph":     RESULTS_ROOT / "periph",
    "periph_nf":  RESULTS_ROOT / "periph-nf",
}

# Explicit probe-stage confusion matrices used for the paper figures.
CM_PATHS = {
    name: run_dir / "probe_confusion_matrix.csv"
    for name, run_dir in RUNS.items()
}

ORDER = ["baseline", "fovea_gaze", "periph", "periph_nf"]
LABEL = {
    "baseline": "Baseline",
    "fovea_gaze": "Fovea-Gaze",
    "periph": "Periph",
    "periph_nf": "Periph-NF",
}

print("RESULTS_ROOT:", RESULTS_ROOT)
print("OUTPUT_DIR: ", OUTPUT_DIR)


## Shared loaders and validation

In [ ]:
# -----------------------------------------------------------------------------
# Linear-probe history helpers
# -----------------------------------------------------------------------------
PROBE_STAGE_NAMES = {"probe", "linear_probe", "linear-probe", "lp", "linearprobe"}

CAND = {
    "x": ["stage_epoch", "epoch", "global_epoch", "step"],
    "eval_macro_f1": ["eval_macro_f1", "macro_f1", "eval_macroF1", "macroF1", "eval_f1", "macro_f1_eval"],
    "eval_acc": ["eval_acc", "acc", "eval_accuracy", "accuracy", "eval_top1", "top1"],
    "eval_loss": ["eval_loss", "loss", "eval_ce", "ce", "val_loss", "valid_loss"],
    "train_loss": ["train_loss", "loss_train", "train_ce", "ce_train"],
    "train_acc": ["train_acc", "acc_train", "train_accuracy", "accuracy_train", "train_top1", "top1_train"],
}

def find_history_json(run_dir: Path):
    direct = run_dir / "history.json"
    if direct.is_file():
        return direct
    cands = list(run_dir.rglob("history.json")) if run_dir.is_dir() else []
    if not cands:
        return None
    return max(cands, key=lambda p: p.stat().st_mtime)

def get_stage(entry: dict) -> str:
    s = entry.get("stage") or entry.get("stage_name") or entry.get("phase") or ""
    return str(s).strip().lower()

def pick_key(entries, candidates):
    keys = set()
    for e in entries:
        keys.update(e.keys())
    for k in candidates:
        if k in keys:
            return k
    lower = {k.lower(): k for k in keys}
    for want in candidates:
        wl = want.lower()
        for kl, orig in lower.items():
            if wl in kl:
                return orig
    return None

def extract_series(entries, y_key):
    if not entries:
        return None
    x_key = pick_key(entries, CAND["x"])
    if x_key is None:
        x = np.arange(1, len(entries) + 1, dtype=float)
    else:
        x = np.array([e.get(x_key, np.nan) for e in entries], dtype=float)
    y = np.array([e.get(y_key, np.nan) for e in entries], dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 2:
        return None
    o = np.argsort(x)
    return x[o], y[o]

def load_probe(run_dir: Path):
    hp = find_history_json(run_dir)
    if hp is None:
        return None, None
    with hp.open("r", encoding="utf-8") as f:
        hist = json.load(f)
    if not isinstance(hist, list) or not hist:
        return None, hp
    probe = [e for e in hist if isinstance(e, dict) and get_stage(e) in PROBE_STAGE_NAMES]
    return probe, hp

probe_entries = {}
hist_paths = {}
for name, run_dir in RUNS.items():
    probe, hp = load_probe(run_dir)
    hist_paths[name] = hp
    if not probe:
        raise FileNotFoundError(
            f"{LABEL[name]}: no linear-probe entries found. history.json={hp}"
        )
    probe_entries[name] = probe
    print(f"[OK] {LABEL[name]}: {len(probe)} probe rows from {hp}")

# Paper figure uses 20 probe epochs. Fail loudly if the wrong stage/history is supplied.
for name, entries in probe_entries.items():
    if len(entries) != 20:
        raise ValueError(
            f"{LABEL[name]} has {len(entries)} probe rows; expected 20 for the paper figure."
        )

# -----------------------------------------------------------------------------
# Confusion-matrix helpers
# -----------------------------------------------------------------------------
def load_cm_csv(path: Path):
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    if df.shape[1] == 0:
        raise ValueError(f"Empty CSV: {path}")
    true_col = df.columns[0]
    true_labels = df[true_col].astype(str).tolist()
    pred_labels = [str(c) for c in df.columns[1:]]
    mat = df.iloc[:, 1:].to_numpy(dtype=float)
    return true_labels, pred_labels, mat

def align_cms(cms):
    common_true = list(cms[next(iter(cms))][0])
    common_pred = list(cms[next(iter(cms))][1])
    for name, (t, p, m) in list(cms.items()):
        if t != common_true or p != common_pred:
            df = pd.DataFrame(m, index=t, columns=p)
            df = df.loc[common_true, common_pred]
            cms[name] = (common_true, common_pred, df.to_numpy(dtype=float))
    return cms, common_true

def row_normalize(mat: np.ndarray) -> np.ndarray:
    denom = mat.sum(axis=1, keepdims=True)
    denom = np.where(denom == 0, 1.0, denom)
    return mat / denom

cms = {name: load_cm_csv(path) for name, path in CM_PATHS.items()}
cms, label_order = align_cms(cms)

if len(label_order) != 17:
    raise ValueError(f"Expected 17 in-domain classes; found {len(label_order)}")

print("[OK] probe confusion matrices loaded and aligned; classes =", len(label_order))


## Figure 2B — In-domain linear-probe curves

This is the final two-panel source PDF used for the in-domain portion of Figure 2: validation loss and Macro-F1 across the 20 **probe** epochs.

In [ ]:
# ============================
# In-Domain: ONE portrait PDF (Loss + Macro-F1)
# ============================

OUT_PDF = OUTPUT_DIR / "indomain_probe_loss_and_macroF1_portrait.pdf"

FIGSIZE_PORTRAIT = (6.6, 7.8)
FS_TITLE  = 16
FS_LABEL  = 14
FS_TICKS  = 12
FS_LEGEND = 12
LW_LINE   = 2.0

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": FS_TICKS,
    "axes.titlesize": FS_TITLE,
    "axes.labelsize": FS_LABEL,
    "xtick.labelsize": FS_TICKS,
    "ytick.labelsize": FS_TICKS,
    "legend.fontsize": FS_LEGEND,
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "savefig.pad_inches": 0.02,
    "axes.xmargin": 0.0,
})

COLOR = {
    "baseline":   "tab:blue",
    "fovea_gaze": "tab:orange",
    "periph":     "tab:green",
    "periph_nf":  "tab:red",
}

XMIN, XMAX = 1.0, 20.0

def resolve_shared_key(metric_name):
    candidates = CAND[metric_name]
    for k in candidates:
        ok = True
        for entries in probe_entries.values():
            keys = set().union(*[e.keys() for e in entries])
            if k not in keys:
                ok = False
                break
        if ok:
            return k
    return None

shared_loss_key = resolve_shared_key("eval_loss")
shared_f1_key   = resolve_shared_key("eval_macro_f1")
print("[INFO] shared keys:", {"eval_loss": shared_loss_key, "eval_macro_f1": shared_f1_key})

def plot_metric_on_ax(ax, metric_name, ylabel, title, shared_key, want_legend=False):
    any_curve = False
    for name in ORDER:
        entries = probe_entries[name]
        y_key = shared_key or pick_key(entries, CAND[metric_name])
        if y_key is None:
            raise KeyError(f"{LABEL[name]}: cannot find key for {metric_name}")
        series = extract_series(entries, y_key)
        if series is None:
            raise ValueError(f"{LABEL[name]}: insufficient data for {y_key}")
        x, y = series
        ax.plot(
            x, y,
            color=COLOR[name],
            linewidth=LW_LINE,
            alpha=0.90,
            label=LABEL[name],
        )
        any_curve = True

    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.30)
    ax.set_xlim(XMIN, XMAX)
    ax.margins(x=0)

    if want_legend and any_curve:
        ax.legend(loc="upper right", frameon=True, framealpha=0.9, borderpad=0.3)

fig, axes = plt.subplots(2, 1, figsize=FIGSIZE_PORTRAIT, sharex=True)

plot_metric_on_ax(
    axes[0],
    metric_name="eval_loss",
    ylabel="Eval loss",
    title="In-domain linear probe: Eval loss",
    shared_key=shared_loss_key,
    want_legend=True,
)

plot_metric_on_ax(
    axes[1],
    metric_name="eval_macro_f1",
    ylabel="Eval macro-F1",
    title="In-domain linear probe: Eval macro-F1",
    shared_key=shared_f1_key,
    want_legend=False,
)

axes[1].set_xlabel("Probe epoch")
fig.tight_layout()
fig.savefig(OUT_PDF, format="pdf")
plt.close(fig)

print("[OK] wrote:", OUT_PDF)


## Supplementary Figure 1 — class-wise ΔF1 with bootstrap CIs

The confusion matrices are resampled **row-wise with a multinomial distribution**, conditional on each observed true-class count.

In [ ]:
def f1_per_class_from_cm(cm):
    cm = np.asarray(cm, dtype=float)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    denom = (2 * tp + fp + fn)
    return np.where(denom > 0, (2 * tp) / denom, np.nan)

def simulate_cm_from_rows(cm, rng):
    cm = np.asarray(cm, dtype=float)
    out = np.zeros_like(cm)
    row_sums = cm.sum(axis=1)
    for i in range(cm.shape[0]):
        n = int(round(row_sums[i]))
        if n <= 0:
            continue
        p = cm[i] / row_sums[i]
        p = np.clip(p, 0, 1)
        p = p / p.sum()
        out[i] = rng.multinomial(n=n, pvals=p)
    return out

def bootstrap_delta_f1(cms, labels, a, b, n_boot=1000, seed=1337):
    rng = np.random.default_rng(seed)
    cm_a = cms[a][2]
    cm_b = cms[b][2]

    f1_a_obs = f1_per_class_from_cm(cm_a)
    f1_b_obs = f1_per_class_from_cm(cm_b)
    delta_obs = f1_a_obs - f1_b_obs

    deltas = []
    for _ in range(n_boot):
        boot_a = simulate_cm_from_rows(cm_a, rng)
        boot_b = simulate_cm_from_rows(cm_b, rng)
        f1_a = f1_per_class_from_cm(boot_a)
        f1_b = f1_per_class_from_cm(boot_b)
        deltas.append(f1_a - f1_b)
    deltas = np.stack(deltas, axis=0)

    ci_low = np.nanpercentile(deltas, 2.5, axis=0)
    ci_high = np.nanpercentile(deltas, 97.5, axis=0)

    return pd.DataFrame({
        "Label": labels,
        "delta_obs": delta_obs,
        "ci_low": ci_low,
        "ci_high": ci_high,
    })

ci_fovea_base = bootstrap_delta_f1(cms, label_order, a="fovea_gaze", b="baseline", n_boot=1000, seed=1337)
ci_fovea_nf   = bootstrap_delta_f1(cms, label_order, a="fovea_gaze", b="periph_nf", n_boot=1000, seed=1337)
ci_fovea_p    = bootstrap_delta_f1(cms, label_order, a="fovea_gaze", b="periph", n_boot=1000, seed=1337)

print("[OK] bootstrap CIs computed")


In [ ]:
# Paper-source Supplementary Figure 1 formatting.
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 12,
})

def _render_plot_to_png(ci_df, title, color, rotate=45, figsize=(14, 5.2), dpi=300):
    d = ci_df.copy()

    labels = d["Label"].astype(str).tolist()
    y  = d["delta_obs"].to_numpy(float)
    lo = d["ci_low"].to_numpy(float)
    hi = d["ci_high"].to_numpy(float)

    x = np.arange(len(y))
    yerr = np.vstack([y - lo, hi - y])

    max_abs = float(np.nanmax(np.abs(np.r_[lo, hi, y])))
    pad = 0.12 * (max_abs + 1e-9)
    ymin, ymax = -(max_abs + pad), (max_abs + pad)

    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(x, y, color=color, linewidth=0)
    ax.errorbar(x, y, yerr=yerr, fmt="none", ecolor="black", elinewidth=1, capsize=3)
    ax.axhline(0, linewidth=1)

    ax.set_title(title, pad=10)
    ax.set_ylabel("ΔF1 (95% CI)")
    ax.set_xlim(-0.6, len(y) - 0.4)
    ax.set_ylim(ymin, ymax)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=rotate, ha="right", rotation_mode="anchor")
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    fig.subplots_adjust(bottom=0.40)

    tmp_path = OUTPUT_DIR / "_tmp_f1_delta_plot.png"
    fig.savefig(tmp_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return Image.open(tmp_path).convert("RGB")

def stitch_vertical(images, pad_px=30, bg=(255, 255, 255)):
    widths = [im.width for im in images]
    max_w = max(widths)
    total_h = sum(im.height for im in images) + pad_px * (len(images) - 1)

    canvas = Image.new("RGB", (max_w, total_h), bg)
    y = 0
    for im in images:
        x = (max_w - im.width) // 2
        canvas.paste(im, (x, y))
        y += im.height + pad_px
    return canvas

img1 = _render_plot_to_png(
    ci_fovea_base,
    title="Fovea > Base (ΔF1) with 95% parametric bootstrap CI",
    color="#1f77b4",
    rotate=45,
)
img2 = _render_plot_to_png(
    ci_fovea_nf,
    title="Fovea > Periph-NF (ΔF1) with 95% parametric bootstrap CI",
    color="#ff7f0e",
    rotate=45,
)
img3 = _render_plot_to_png(
    ci_fovea_p,
    title="Fovea > Periph (ΔF1) with 95% parametric bootstrap CI",
    color="#2ca02c",
    rotate=45,
)

combined = stitch_vertical([img1, img2, img3], pad_px=40)
combined_png = OUTPUT_DIR / "f1_delta_bootstrap_CI_all_three.png"
combined_pdf = OUTPUT_DIR / "f1_delta_bootstrap_CI_all_three.pdf"
combined.save(combined_png)
combined.save(combined_pdf, "PDF", resolution=300)

# Close PIL handles and remove the temporary raster.
for im in (img1, img2, img3):
    im.close()
tmp_path = OUTPUT_DIR / "_tmp_f1_delta_plot.png"
if tmp_path.exists():
    tmp_path.unlink()

print("[OK] wrote:", combined_png)
print("[OK] wrote:", combined_pdf)


## Supplementary Figure 3 — row-normalized confusion matrices


In [ ]:
def set_confusion_paper_rc():
    plt.rcParams.update({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.dpi": 200,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.03,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

def plot_2x2_confusions(mats_norm, labels, titles, out_pdf, cmap="viridis"):
    set_confusion_paper_rc()
    n = len(labels)

    fig, axes = plt.subplots(2, 2, figsize=(11.0, 9.5), constrained_layout=True)
    axes = axes.ravel()

    vmin, vmax = 0.0, 1.0
    ims = []
    for ax, mat, title in zip(axes, mats_norm, titles):
        im = ax.imshow(
            mat,
            aspect="equal",
            interpolation="nearest",
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
        )
        ims.append(im)
        ax.set_title(title)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_xticks(np.arange(n))
        ax.set_yticks(np.arange(n))
        ax.set_xticklabels(labels, rotation=90, ha="center")
        ax.set_yticklabels(labels)
        ax.tick_params(axis="both", which="both", length=2)

    cbar = fig.colorbar(ims[0], ax=axes, fraction=0.035, pad=0.02)
    cbar.set_label("Row-normalized proportion (rows sum to 1)")

    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)

conf_order = ["baseline", "fovea_gaze", "periph_nf", "periph"]
mats_norm = [row_normalize(cms[name][2]) for name in conf_order]
titles = [LABEL[name] for name in conf_order]

supp3_pdf = OUTPUT_DIR / "confusion_matrices_2x2_row_norm.pdf"
plot_2x2_confusions(mats_norm, label_order, titles, supp3_pdf)
print("[OK] wrote:", supp3_pdf)


## Figure 3 — MDS of class confusion profiles

The MDS computation final analysis logic:

1. row-normalize each 17×17 confusion matrix,
2. compute pairwise Jensen–Shannon distances between class confusion profiles,
3. run 2-D metric MDS with `random_state=0`, `n_init=10`, and `max_iter=2000`,
4. plot the four panels using the final class-color palette and label-repulsion settings.

In [ ]:
# adjustText was used for the final MDS label placement.
# Install it once in your environment if needed:
#   pip install adjustText
try:
    from adjustText import adjust_text
except ImportError as exc:
    raise ImportError(
        "Figure 3 requires the 'adjustText' package. Install it with: pip install adjustText"
    ) from exc

from sklearn.manifold import MDS
from scipy.spatial.distance import jensenshannon

def js_distance_matrix(P: np.ndarray, eps=1e-12) -> np.ndarray:
    P = np.clip(P, eps, 1.0)
    P = P / P.sum(axis=1, keepdims=True)
    n = P.shape[0]
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            d = jensenshannon(P[i], P[j])
            D[i, j] = D[j, i] = float(d)
    return D

def run_mds(D: np.ndarray, seed=0):
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        random_state=seed,
        n_init=10,
        max_iter=2000,
        normalized_stress="auto",
    )
    X = mds.fit_transform(D)
    return X, getattr(mds, "stress_", None)

SEED = 0
coords = {}
mds_stress = {}
for name in ORDER:
    P = row_normalize(cms[name][2])
    D = js_distance_matrix(P)
    X, stress = run_mds(D, seed=SEED)
    coords[name] = X
    mds_stress[name] = stress
    print(f"[OK] {LABEL[name]} MDS stress: {stress:.6f}" if stress is not None else f"[OK] {LABEL[name]}")


In [ ]:
# Final Figure 3 plotting style retained from the original final cell.
# The rcParams below reproduce the paper-oriented MDS state set immediately
# before the final plotting cell in the original notebook.
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

mds_order = ["baseline", "fovea_gaze", "periph", "periph_nf"]
mds_titles = {
    "baseline": "Baseline",
    "fovea_gaze": "Fovea-Gaze",
    "periph": "Periph",
    "periph_nf": "Periph-NF",
}

# Fixed class colors across all panels (same palette/index logic as final cell).
cmaps = [plt.get_cmap("tab20b"), plt.get_cmap("tab20c")]
palette = [cmaps[0](i) for i in range(20)] + [cmaps[1](i) for i in range(20)]
candidate_idxs = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18,
                  20, 22, 24, 26, 28, 30, 32]
class_colors = [palette[i] for i in candidate_idxs]

def robust_limits(X, lo_q=2, hi_q=98, pad_frac=0.65, min_span=0.90):
    (xmin_q, ymin_q) = np.percentile(X, lo_q, axis=0)
    (xmax_q, ymax_q) = np.percentile(X, hi_q, axis=0)
    xr = max(1e-9, xmax_q - xmin_q)
    yr = max(1e-9, ymax_q - ymin_q)

    x0 = xmin_q - pad_frac * xr
    x1 = xmax_q + pad_frac * xr
    y0 = ymin_q - pad_frac * yr
    y1 = ymax_q + pad_frac * yr

    if (x1 - x0) < min_span:
        c = 0.5 * (x0 + x1)
        x0, x1 = c - 0.5 * min_span, c + 0.5 * min_span
    if (y1 - y0) < min_span:
        c = 0.5 * (y0 + y1)
        y0, y1 = c - 0.5 * min_span, c + 0.5 * min_span

    return x0, x1, y0, y1

fig, axes = plt.subplots(2, 2, figsize=(6.9, 5.4))
axes = axes.ravel()

for ax, name in zip(axes, mds_order):
    X = coords[name]

    if name == "fovea_gaze":
        x0, x1, y0, y1 = robust_limits(X, pad_frac=0.95, min_span=1.10)
        fs = 7.0
        exp_pts, exp_txt = (3.2, 4.2), (2.8, 3.8)
        f_pts, f_txt = 1.6, 2.6
        lim = 4000
        arrow = dict(arrowstyle="-", lw=0.25, alpha=0.55)
        s_pts = 16
    else:
        x0, x1, y0, y1 = robust_limits(X, pad_frac=0.65, min_span=0.90)
        fs = 7.5
        exp_pts, exp_txt = (1.6, 2.0), (1.4, 1.8)
        f_pts, f_txt = 0.5, 0.9
        lim = 700
        arrow = dict(arrowstyle="-", lw=0.30, alpha=0.75)
        s_pts = 18

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.scatter(X[:, 0], X[:, 1], s=s_pts, c=class_colors, edgecolors="none", alpha=0.95)

    texts = []
    for i, lab in enumerate(label_order):
        texts.append(ax.text(X[i, 0], X[i, 1], lab, fontsize=fs, color=class_colors[i]))

    adjust_text(
        texts,
        x=X[:, 0],
        y=X[:, 0] * 0 + X[:, 1],
        ax=ax,
        expand_points=exp_pts,
        expand_text=exp_txt,
        force_points=f_pts,
        force_text=f_txt,
        lim=lim,
        only_move={"text": "xy"},
        arrowprops=arrow,
    )

    ax.set_title(mds_titles[name], pad=3)
    ax.set_xlabel("MDS-1", labelpad=1)
    ax.set_ylabel("MDS-2", labelpad=1)
    ax.tick_params(length=2, pad=1)

fig.suptitle("MDS of class confusion profiles (JS distance)", y=0.995, fontsize=11)
fig.tight_layout(pad=0.6)

mds_pdf = OUTPUT_DIR / "mds_2x2_grid_procrustes.pdf"
fig.savefig(mds_pdf, format="pdf", bbox_inches="tight")
plt.close(fig)
print("[OK] wrote:", mds_pdf)


## Outputs

Note that a successful run produces only the final in-domain figure source files:

```text
indomain_probe_loss_and_macroF1_portrait.pdf
f1_delta_bootstrap_CI_all_three.png
f1_delta_bootstrap_CI_all_three.pdf
confusion_matrices_2x2_row_norm.pdf
mds_2x2_grid_procrustes.pdf
```

In [ ]:
expected_outputs = [
    "indomain_probe_loss_and_macroF1_portrait.pdf",
    "f1_delta_bootstrap_CI_all_three.png",
    "f1_delta_bootstrap_CI_all_three.pdf",
    "confusion_matrices_2x2_row_norm.pdf",
    "mds_2x2_grid_procrustes.pdf",
]
for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(("[OK]" if path.exists() else "[MISSING]"), path)
